In [ ]:
# Plant Analyzer for iPhone/Jupyter (Carnets) - Fixed Version
# This version has proper imports and error handling for Carnets

import numpy as np
from PIL import Image
import io
import base64
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from collections import Counter

class iPhonePlantAnalyzer:
    def __init__(self):
        self.analysis_results = {}
        
    def analyze_image(self, image_array):
        """Analyze plant image from numpy array"""
        self.pixels = image_array
        self.analysis_results = {}
        
        # Show processing message
        display(HTML("<div style='text-align: center; color: #2e7d32;'><h3>🔍 Analyzing Plant Health...</h3></div>"))
        
        # Run all analyses
        self.extract_dominant_colors()
        self.analyze_green_health()
        self.detect_discoloration()
        self.analyze_plant_structure()
        
        return self.generate_friendly_report()
    
    def extract_dominant_colors(self, k=5):
        """Extract dominant colors"""
        pixels = self.pixels.reshape(-1, 3)
        
        color_groups = {}
        for pixel in pixels[:5000]:
            pixel_tuple = tuple(int(x) for x in pixel)
            key = tuple((pixel_tuple[i] // 32) * 32 for i in range(3))
            color_groups[key] = color_groups.get(key, 0) + 1
        
        top_colors = sorted(color_groups.items(), key=lambda x: x[1], reverse=True)[:k]
        total = sum(count for _, count in top_colors)
        
        dominant_colors = []
        for color, count in top_colors:
            percentage = (count / total) * 100
            dominant_colors.append({
                'rgb': color,
                'percentage': round(percentage, 2)
            })
        
        self.analysis_results['dominant_colors'] = dominant_colors
        return dominant_colors
    
    def analyze_green_health(self):
        """Analyze plant health based on green color presence"""
        pixels = self.pixels.reshape(-1, 3)
        
        green_pixels = 0
        green_intensities = []
        
        for pixel in pixels[:3000]:
            r, g, b = int(pixel[0]), int(pixel[1]), int(pixel[2])
            if g > r and g > b and g > 50:
                green_pixels += 1
                green_intensities.append(g)
        
        total_sampled = min(3000, len(pixels))
        green_percentage = (green_pixels / total_sampled) * 100
        avg_green_intensity = np.mean(green_intensities) if green_intensities else 0
        
        health_indicators = {
            'green_coverage_percentage': round(green_percentage, 2),
            'avg_green_intensity': round(float(avg_green_intensity), 2),
            'health_score': min(100, round(float((green_percentage * avg_green_intensity) / 255), 2))
        }
        
        self.analysis_results['health_analysis'] = health_indicators
        return health_indicators
    
    def detect_discoloration(self):
        """Detect yellow/brown areas"""
        pixels = self.pixels.reshape(-1, 3)
        
        yellow_count = 0
        brown_count = 0
        
        for pixel in pixels[:3000]:
            r, g, b = int(pixel[0]), int(pixel[1]), int(pixel[2])
            
            if r > 150 and g > 150 and b < 100:
                yellow_count += 1
            
            if 100 < r < 200 and g < r and b < 100:
                brown_count += 1
        
        total_sampled = min(3000, len(pixels))
        yellow_percentage = (yellow_count / total_sampled) * 100
        brown_percentage = (brown_count / total_sampled) * 100
        
        discoloration = {
            'yellow_percentage': round(yellow_percentage, 2),
            'brown_percentage': round(brown_percentage, 2),
            'total_discoloration': round(yellow_percentage + brown_percentage, 2)
        }
        
        self.analysis_results['discoloration_analysis'] = discoloration
        return discoloration
    
    def analyze_plant_structure(self):
        """Analyze basic plant structure"""
        height, width = self.pixels.shape[:2]
        aspect_ratio = width / height
        
        structure_analysis = {
            'image_dimensions': f"{width}x{height}",
            'aspect_ratio': round(float(aspect_ratio), 2),
            'size_category': 'large' if max(width, height) > 1000 else 'medium' if max(width, height) > 500 else 'small'
        }
        
        self.analysis_results['structure_analysis'] = structure_analysis
        return structure_analysis
    
    def generate_friendly_report(self):
        """Generate a user-friendly HTML report"""
        health = self.analysis_results['health_analysis']
        discoloration = self.analysis_results['discoloration_analysis']
        colors = self.analysis_results['dominant_colors']
        structure = self.analysis_results['structure_analysis']
        
        # Determine health status with emoji
        if health['health_score'] > 70:
            health_status = "🌿 Likely Healthy"
            health_color = "green"
        elif health['health_score'] > 40:
            health_status = "⚠️ Moderately Healthy"
            health_color = "orange"
        else:
            health_status = "❌ Potentially Unhealthy"
            health_color = "red"
        
        # Generate recommendations
        recommendations = []
        if health['health_score'] > 70:
            recommendations.append("✅ Your plant appears healthy! Continue current care routine.")
        else:
            recommendations.append("💧 Check your watering schedule")
            recommendations.append("🌱 Consider plant-appropriate fertilizer")
            
        if discoloration['yellow_percentage'] > 10:
            recommendations.append("⚠️ Yellow leaves may indicate overwatering")
            
        if discoloration['brown_percentage'] > 5:
            recommendations.append("🔥 Brown tips could mean too much sun or under-watering")
            
        if health['green_coverage_percentage'] < 30:
            recommendations.append("📷 For better analysis, take a closer photo focused on the plant")
        
        if len(recommendations) == 0:
            recommendations.append("🌿 No specific issues detected. Maintain regular plant care.")
        
        # Create HTML report
        html_report = f"""
        <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; max-width: 600px; margin: 20px auto; padding: 20px; background: #f8f9fa; border-radius: 15px;">
            <h1 style="color: #2e7d32; text-align: center; margin-bottom: 30px;">🌱 Plant Health Analysis</h1>
            
            <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h2 style="color: #2e7d32; margin-top: 0;">Overall Status: <span style="color: {health_color};">{health_status}</span></h2>
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px;">
                    <div style="text-align: center; padding: 10px; background: #e8f5e8; border-radius: 8px;">
                        <h3 style="margin: 0; color: #2e7d32;">Health Score</h3>
                        <p style="font-size: 24px; font-weight: bold; margin: 5px 0; color: {health_color};">{health['health_score']}/100</p>
                    </div>
                    <div style="text-align: center; padding: 10px; background: #e8f5e8; border-radius: 8px;">
                        <h3 style="margin: 0; color: #2e7d32;">Green Coverage</h3>
                        <p style="font-size: 24px; font-weight: bold; margin: 5px 0; color: #2e7d32;">{health['green_coverage_percentage']}%</p>
                    </div>
                </div>
            </div>
            
            <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h3 style="color: #2e7d32;">📊 Detailed Analysis</h3>
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
                    <div><strong>Green Intensity:</strong> {health['avg_green_intensity']}/255</div>
                    <div><strong>Yellow Areas:</strong> {discoloration['yellow_percentage']}%</div>
                    <div><strong>Brown Areas:</strong> {discoloration['brown_percentage']}%</div>
                    <div><strong>Image Size:</strong> {structure['image_dimensions']}</div>
                </div>
            </div>
            
            <div style="background: white; padding: 20px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h3 style="color: #2e7d32;">💡 Care Recommendations</h3>
                <ul style="padding-left: 20px;">
        """
        
        for rec in recommendations:
            html_report += f'<li style="margin-bottom: 8px;">{rec}</li>'
        
        html_report += """
                </ul>
            </div>
            
            <div style="text-align: center; color: #666; font-size: 12px; margin-top: 20px;">
                Analysis completed with Plant Analyzer 🌿
            </div>
        </div>
        """
        
        return html_report

# Create global analyzer instance
analyzer = iPhonePlantAnalyzer()

def simple_plant_analyzer():
    """Simplified plant analyzer that works reliably in Carnets"""
    try:
        clear_output()
        
        # Display welcome message
        welcome_html = """
        <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; 
                    max-width: 600px; margin: 20px auto; padding: 20px; text-align: center;">
            <h1 style="color: #2e7d32;">🌱 Plant Health Analyzer</h1>
            <p style="color: #666; margin-bottom: 30px;">Upload a photo of your plant to analyze its health</p>
            
            <div style="background: white; padding: 30px; border-radius: 15px; border: 2px dashed #2e7d32; margin: 20px 0;">
                <h3 style="color: #2e7d32; margin-top: 0;">📷 How to Upload:</h3>
                <ol style="text-align: left; display: inline-block;">
                    <li>Click the <strong>Upload</strong> button below</li>
                    <li>Select <strong>Photo Library</strong> or <strong>Take Photo</strong></li>
                    <li>Choose your plant photo</li>
                    <li>Wait for analysis results</li>
                </ol>
            </div>
        </div>
        """
        display(HTML(welcome_html))
        
        # Use ipywidgets for file upload
        import ipywidgets as widgets
        
        # Create file upload widget
        uploader = widgets.FileUpload(
            accept='image/*',
            multiple=False,
            description='Upload Photo',
            style={'button_color': '#2e7d32'}
        )
        
        # Create output area for results
        output_area = widgets.Output()
        
        def handle_upload(change):
            """Handle file upload and process image"""
            with output_area:
                clear_output()
                
                if uploader.value:
                    # Get the uploaded file
                    uploaded_file = list(uploader.value.values())[0]
                    image_data = uploaded_file['content']
                    
                    # Show processing message
                    display(HTML("<div style='text-align: center;'><h3>🔄 Processing Image...</h3></div>"))
                    
                    try:
                        # Process image
                        image = Image.open(io.BytesIO(image_data))
                        image_array = np.array(image)
                        
                        # Display the uploaded image
                        plt.figure(figsize=(8, 6))
                        plt.imshow(image_array)
                        plt.axis('off')
                        plt.title('Your Plant Photo', fontsize=16, pad=20)
                        plt.show()
                        
                        # Analyze the image
                        html_report = analyzer.analyze_image(image_array)
                        display(HTML(html_report))
                        
                        # Show success message
                        display(HTML("""
                        <div style="text-align: center; margin: 30px; padding: 20px; background: #e8f5e8; border-radius: 10px;">
                            <h3 style="color: #2e7d32;">✅ Analysis Complete!</h3>
                            <p>To analyze another plant, run this cell again.</p>
                        </div>
                        """))
                        
                    except Exception as e:
                        display(HTML(f"""
                        <div style="background: #ffebee; color: #c62828; padding: 20px; border-radius: 10px; text-align: center;">
                            <h3>❌ Error Processing Image</h3>
                            <p>Please try again with a different photo.</p>
                            <p><small>Error: {str(e)}</small></p>
                        </div>
                        """))
        
        # Connect the upload handler
        uploader.observe(handle_upload, names='value')
        
        # Display the uploader and output area
        display(uploader)
        display(output_area)
        
    except Exception as e:
        # Fallback if ipywidgets fails
        display(HTML(f"""
        <div style="background: #fff3cd; padding: 20px; border-radius: 10px; margin: 20px 0;">
            <h3>⚠️ Upload Not Available</h3>
            <p>File upload is not working in this environment.</p>
            <p>Error: {str(e)}</p>
            <p>Please try the demo analysis instead.</p>
        </div>
        """))

def quick_demo_analysis():
    """Show a demo analysis without needing to upload a photo"""
    clear_output()
    
    display(HTML("""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; 
                max-width: 600px; margin: 20px auto; padding: 20px; text-align: center;">
        <h1 style="color: #2e7d32;">🌱 Plant Health Analyzer Demo</h1>
        <p>Showing sample analysis of a healthy plant</p>
    </div>
    """))
    
    # Create a sample healthy plant image (mostly green)
    height, width = 400, 600
    sample_image = np.zeros((height, width, 3), dtype=np.uint8)
    
    # Make it look like a healthy plant (mostly green with some variation)
    for i in range(height):
        for j in range(width):
            # Create green gradient with some noise
            green_value = 150 + int(70 * (i / height)) + np.random.randint(-20, 20)
            sample_image[i, j] = [
                max(0, min(255, green_value - 60 + np.random.randint(-10, 10))),  # Red
                max(0, min(255, green_value)),  # Green (dominant)
                max(0, min(255, green_value - 80 + np.random.randint(-10, 10)))   # Blue
            ]
    
    # Display sample image
    plt.figure(figsize=(8, 6))
    plt.imshow(sample_image)
    plt.axis('off')
    plt.title('Sample Healthy Plant', fontsize=16, pad=20)
    plt.show()
    
    # Analyze and display results
    html_report = analyzer.analyze_image(sample_image)
    display(HTML(html_report))
    
    display(HTML("""
    <div style="text-align: center; margin: 30px; padding: 20px; background: #e3f2fd; border-radius: 10px;">
        <h3 style="color: #1976d2;">Ready to Analyze Your Plant?</h3>
        <p>Run <code>simple_plant_analyzer()</code> to upload your own plant photos!</p>
    </div>
    """))

def show_help():
    """Display help instructions"""
    help_html = """
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; 
                max-width: 600px; margin: 20px auto; padding: 20px; background: #f5f5f5; border-radius: 10px;">
        <h2 style="color: #2e7d32;">🌿 Plant Analyzer Help</h2>
        
        <h3 style="color: #2e7d32;">Available Functions:</h3>
        <ul>
            <li><code>simple_plant_analyzer()</code> - Upload and analyze your plant photos</li>
            <li><code>quick_demo_analysis()</code> - See a sample analysis</li>
            <li><code>show_help()</code> - Show this help message</li>
        </ul>
        
        <h3 style="color: #2e7d32;">Tips for Best Results:</h3>
        <ul>
            <li>📸 Take photos in good natural light</li>
            <li>🌿 Focus on the plant leaves</li>
            <li>⚪ Use a plain background</li>
            <li>🔍 Get close enough to see leaf details</li>
        </ul>
        
        <h3 style="color: #2e7d32;">Troubleshooting:</h3>
        <ul>
            <li>If upload fails, try a smaller image</li>
            <li>Make sure to allow photo access when prompted</li>
            <li>Try the demo first to see how it works</li>
        </ul>
    </div>
    """
    display(HTML(help_html))

# Initialize and show available functions
print("🌱 Plant Analyzer for iPhone Ready!")
print("\nAvailable functions:")
print("• simple_plant_analyzer()  - Upload and analyze your plant photos")
print("• quick_demo_analysis()    - See a sample analysis") 
print("• show_help()              - Get help and instructions")